# Notebook 1: Train CNN on CIFAR-10

**Goal:** Train a CNN classifier that will be the target for adversarial attacks.

**Output:** Saved model weights in `../results/model.pth`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import sys
sys.path.append('../')

from src.model import CNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load CIFAR-10 dataset
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False, num_workers=2)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f'Train size: {len(trainset)} | Test size: {len(testset)}')

In [ ]:
# Visualize some samples
images, labels = next(iter(trainloader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    img = images[i].permute(1,2,0).numpy()
    img = img * 0.2 + 0.45  # unnormalize roughly
    ax.imshow(img.clip(0,1))
    ax.set_title(classes[labels[i]])
    ax.axis('off')
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150)
plt.show()

In [ ]:
# Initialize model, loss, optimizer
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Training loop
EPOCHS = 30
train_losses, test_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Evaluate on test set
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    avg_loss = running_loss / len(trainloader)
    train_losses.append(avg_loss)
    test_accs.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

torch.save(model.state_dict(), '../results/model.pth')
print('Model saved!')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax2.plot(test_accs)
ax2.set_title('Test Accuracy')
ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('../results/figures/training_curves.png', dpi=150)
plt.show()
print(f'Final test accuracy: {test_accs[-1]:.4f}')